# Create Silver Database & Ingest CSVs as Delta Tables

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
print("Silver database created")

Silver database created


## Load & Clean All Bronze Tables

In [0]:
from pyspark.sql.functions import col, to_date, concat_ws, round

# Load all bronze tables
categories    = spark.table("bronze.categories")
customers     = spark.table("bronze.customers")
employees     = spark.table("bronze.employees")
orders        = spark.table("bronze.orders")
ordersdetails = spark.table("bronze.ordersdetails")
products      = spark.table("bronze.products")
shippers      = spark.table("bronze.shippers")
suppliers     = spark.table("bronze.suppliers")

# Store all DataFrames in a dictionary
tables = {
    "categories": categories,
    "customers": customers,
    "employees": employees,
    "orders": orders,
    "ordersdetails": ordersdetails,
    "products": products,
    "shippers": shippers,
    "suppliers": suppliers
}

# Loop through and print dtypes
for name, df in tables.items():
    print(f"Data types for {name}:")
    print(df.dtypes)
    print("\n")

Data types for categories:
[('CategoryID', 'int'), ('CategoryName', 'string'), ('DescriptionText', 'string')]


Data types for customers:
[('CustomerID', 'int'), ('CustomerName', 'string'), ('ContractName', 'string'), ('Address', 'string'), ('City', 'string'), ('PostalCode', 'string'), ('Country', 'string')]


Data types for employees:
[('EmployeeID', 'int'), ('LastName', 'string'), ('FirstName', 'string'), ('BirthDate', 'date'), ('Photo', 'string'), ('Notes', 'string')]


Data types for orders:
[('OrderID', 'int'), ('CustomerID', 'int'), ('EmployeeID', 'int'), ('OrderDate', 'date'), ('ShipperID', 'int')]


Data types for ordersdetails:
[('OrderDetailID', 'int'), ('OrderID', 'int'), ('ProductID', 'int'), ('Quantity', 'int')]


Data types for products:
[('ProductID', 'int'), ('ProductName', 'string'), ('SuppliersID', 'int'), ('CategoryID', 'int'), ('Unit', 'string'), ('Price', 'double')]


Data types for shippers:
[('ShipperID', 'int'), ('ShipperName', 'string'), ('Phone', 'string')]




In [0]:
# --- Clean categories ---
categories = categories.withColumnRenamed("DescriptionText", "Description")

# --- Clean customers ---
customers = customers.withColumnRenamed("ContractName", "ContactName")

# --- Clean employees ---
employees = (employees
    .drop("Photo", "Notes")
    .withColumn("BirthDate", to_date(col("BirthDate")))
    .withColumn("FullName", concat_ws(" ", col("FirstName"), col("LastName"))))

# --- Clean orders ---
orders = orders.withColumn("OrderDate", to_date(col("OrderDate")))

# --- Clean products ---
products = products.withColumnRenamed("SuppliersID", "SupplierID")

# --- Clean suppliers ---
suppliers = suppliers.withColumnRenamed("SuppliersName", "SupplierName")

print("All bronze tables cleaned")

All bronze tables cleaned


## Build Silver Table: `orders_enriched`

In [0]:
orders_enriched = (
    orders
    # Join customers
    .join(
        customers.select("CustomerID", "CustomerName", "ContactName", "City", "Country"),
        on="CustomerID",
        how="left"
    )
    .withColumnRenamed("City", "CustomerCity")
    .withColumnRenamed("Country", "CustomerCountry")
    # Join employees
    .join(
        employees.select("EmployeeID", "FullName"),
        on="EmployeeID",
        how="left"
    )
    .withColumnRenamed("FullName", "EmployeeName")
    # Join shippers
    .join(
        shippers.select("ShipperID", "ShipperName"),
        on="ShipperID",
        how="left"
    )
    # Drop FK columns
    .drop("CustomerID", "EmployeeID", "ShipperID")
)

# Save as Delta table
(orders_enriched.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.orders_enriched"))

print(f"silver.orders_enriched: {orders_enriched.count()} rows")
orders_enriched.show(3)

silver.orders_enriched: 153 rows
+-------+----------+--------------------+-----------------+--------------+---------------+----------------+----------------+
|OrderID| OrderDate|        CustomerName|      ContactName|  CustomerCity|CustomerCountry|    EmployeeName|     ShipperName|
+-------+----------+--------------------+-----------------+--------------+---------------+----------------+----------------+
|  10248|2024-11-15|              Wilman|  Matti Karttunen|      Helsinki|        Finland| Steven Buchanan|Federal Shipping|
|  10249|2024-11-16|Tradição Hipermer...|Anabela Domingues|     São Paulo|         Brazil|  Michael Suyama|  Speedy Express|
|  10250|2024-11-18|       Hanari Carnes|     Mario Pontes|Rio de Janeiro|         Brazil|Margaret Peacock|  United Package|
+-------+----------+--------------------+-----------------+--------------+---------------+----------------+----------------+
only showing top 3 rows


## Build Silver Table: `order_items_enriched`

In [0]:
order_items_enriched = (
    ordersdetails
    # Join products
    .join(
        products.select("ProductID", "ProductName", "SupplierID", "CategoryID", "Unit", "Price"),
        on="ProductID",
        how="left"
    )
    # Join categories
    .join(
        categories.select("CategoryID", "CategoryName", "Description"),
        on="CategoryID",
        how="left"
    )
    # Join suppliers
    .join(
        suppliers.select("SupplierID", "SupplierName", "City", "Country"),
        on="SupplierID",
        how="left"
    )
    .withColumnRenamed("City", "SupplierCity")
    .withColumnRenamed("Country", "SupplierCountry")
    # Calculate line revenue
    .withColumn("LineRevenue", round(col("Price") * col("Quantity"), 2))
    # Drop FK columns
    .drop("SupplierID", "CategoryID")
)

# Save as Delta table
(order_items_enriched.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.order_items_enriched"))

print(f"silver.order_items_enriched: {order_items_enriched.count()} rows")
order_items_enriched.show(3)

silver.order_items_enriched: 199 rows
+---------+-------------+-------+--------+--------------------+----------------+-----+--------------+--------------------+--------------------+------------+---------------+-----------+
|ProductID|OrderDetailID|OrderID|Quantity|         ProductName|            Unit|Price|  CategoryName|         Description|        SupplierName|SupplierCity|SupplierCountry|LineRevenue|
+---------+-------------+-------+--------+--------------------+----------------+-----+--------------+--------------------+--------------------+------------+---------------+-----------+
|       11|            1|  10248|      12|      Queso Cabrales|       1 kg pkg.| 21.0|Dairy Products|             Cheeses|Cooperativa de Qu...|      Oviedo|          Spain|      252.0|
|       42|            2|  10248|      10|Singaporean Hokki...| 32 - 1 kg pkgs.| 14.0|Grains/Cereals|Breads, crackers,...|        Leka Trading|   Singapore|      Singapore|      140.0|
|       72|            3|  10248|    

# Verify Silver Tables

In [0]:
for table in ["orders_enriched", "order_items_enriched"]:
    df = spark.table(f"silver.{table}")
    print(f"silver.{table}: {df.count()} rows")
    print(f"columns: {df.columns}\n")

silver.orders_enriched: 153 rows
columns: ['OrderID', 'OrderDate', 'CustomerName', 'ContactName', 'CustomerCity', 'CustomerCountry', 'EmployeeName', 'ShipperName']

silver.order_items_enriched: 199 rows
columns: ['ProductID', 'OrderDetailID', 'OrderID', 'Quantity', 'ProductName', 'Unit', 'Price', 'CategoryName', 'Description', 'SupplierName', 'SupplierCity', 'SupplierCountry', 'LineRevenue']

